In [1]:
import sys
sys.path.append('/data/zkl/AgenticIR')
import json
import torch
from llm.depictqa import DepictQA
from llm.mistral import Mistral
from pathlib import Path
from exploration_self_evolve.instructions_list.add_degradations import PairedDataset
from omegaconf import OmegaConf

config = OmegaConf.load("/data/zkl/AgenticIR/dataset/sr.yaml")

In [2]:
dataset = PairedDataset(opt=config.train)
dl_train = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False,
                                           num_workers=1)
depictqa = DepictQA()
mllm = Mistral()
degradations_feature_mapping = {
    "motion blur": "motion blur",
    "defocus blur": "defocus blur",
    "rain": "Rain strips/streams (slender, short); Directionality (disorder, verticality, inclination); Specular highlights: (not obvious, strong reflection in night scenes or lighting); Clarity: (no change, decreased); Mist/fogging effect (not obvious, with fogging)",
    "haze": "Contrast (unchanged, distant objects appear grayish white and nearby objects are relatively clear); Saturation (unchanged, grayish, whitish); Brightness/air light (no change, overall whitening, dark areas become brighter, bright areas become darker); Edge details (unchanged, faded); Halo/halo (no change, point light source has blurred aperture)",
    "dark": "dark",
    "noise": "noise",
    "jpeg compression artifact": "Blocking effect (no change, brightness/chromaticity jump at the boundary, block like structure); Ringing artifacts: (no change, appearing around the edges); Color blending/chroma smearing: (no change, color distortion, appearance of color blocks); Details/textures (no changes, smoother); Spectrum grid: (no variation, periodic texture); Abnormal brightness/contrast: (no change, decreased, resonant bright dark texture)",
    "low resolution": "low resolution"
}

Number of low-quality paths: 3
Number of ground truth paths: 3


In [3]:
from PIL import Image
import tempfile
import os

def split_image_into_quadrants(image_path, temp_dir=None):
    """Split an image into four quadrants and return paths to the saved tiles."""
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")

    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    mid_w, mid_h = width // 2, height // 2

    # Ensure we cover the entire image even when width/height are odd.
    crops = {
        "top_left": (0, 0, mid_w, mid_h),
        "top_right": (mid_w, 0, width, mid_h),
        "bottom_left": (0, mid_h, mid_w, height),
        "bottom_right": (mid_w, mid_h, width, height),
    }

    if temp_dir is None:
        temp_dir = Path(tempfile.mkdtemp(prefix="quadrants_"))
    else:
        temp_dir = Path(temp_dir)
        temp_dir.mkdir(parents=True, exist_ok=True)

    suffix = ".png"
    saved_paths = []
    for name, box in crops.items():
        quadrant = image.crop(box)
        save_path = temp_dir / f"{name}{suffix}"
        quadrant.save(save_path)
        saved_paths.append(str(save_path))

    return saved_paths

In [4]:
### Construct Instruction List
instruction_list = []

levels = ["very low", "low", "medium", "high", "very high"]
corners = ["top_left", "top_right", "bottom_left", "bottom_right"]
for step, data in enumerate(dl_train):
    print(f"Processing image {step+1}/{len(dl_train)}")

    # Get DepictQA degradation analysis in global view
    depictqa_rep_globle = depictqa.query(
        img_path_lst=[Path(data['lq_path'][0])],
        task="eval_degradation",
        degradation=data['degradation_type'][0])[1]

    # Get DepictQA degradation analysis in local quadrants view
    depictqa_rep_local_list = []
    quadrant_paths = split_image_into_quadrants(data['lq_path'][0], temp_dir="/data/zkl/AgenticIR/exploration_self_evolve/instructions_list/temp")
    for i, quadrant_path in enumerate(quadrant_paths):
        depictqa_rep_local = depictqa.query(
            img_path_lst=[Path(quadrant_path)],
            task="eval_degradation",
            degradation=data['degradation_type'][0])[1]
        
        depictqa_rep_local_list.append((corners[i], depictqa_rep_local))
        
    print(depictqa_rep_local_list)

    instruction_list.append({
        "gt_image_path": data['gt_path'][0],
        "lq_image_path": data['lq_path'][0],
        "degradation_info": []
    })

    for degradation_global, severity_global in depictqa_rep_globle:
        instruction_list[-1]["degradation_info"].append({
            "degradation_global": degradation_global,
            "severity_global": severity_global,
            "degradation_local": {}
        })

        for corner, degradation_rep_local in depictqa_rep_local_list:
            print(corner, degradation_rep_local)

            prompt = f"""You are an image quality expert assigned to analyze a specific image that displays a particular type of degradation, identified as "{degradation_rep_local[0][0]}", with an associated severity level of "{degradation_rep_local[0][1]}". Your primary objective is to deliver a precise and focused description of the visual degradation characteristics present in the image, emphasizing only these degradation aspects and intentionally minimizing any direct references to the broader image content. In your analysis, you must incorporate the specified feature "{degradations_feature_mapping[degradation_rep_local[0][0]]}" by crafting one concise and accurate sentence that describes the degradation in a manner consistent with the format relevant to this feature. Immediately following this, provide a second, brief sentence that explains where this specific feature appears in the image, grounding your explanation in the context of the image content to substantiate your assessment. Structure your response clearly and succinctly, ensuring it consists strictly of two sentences: the first sentence should detail solely the visual degradation characteristic, and the second sentence should offer the rationale for the feature’s manifestation, supported by verification from the image content."""

            mllm_rep = mllm.query(
                img_path_lst=[Path(data['lq_path'][0])],
                prompt=prompt
            )[1].strip()

            instruction_list[-1]["degradation_info"][-1]["degradation_local"].update({
                f"severity_{corner}": f"{degradation_rep_local[0][1]}",
                f"detail_desc_{corner}": mllm_rep
            })

Processing image 1/3
[('top_left', [('haze', 'medium')]), ('top_right', [('haze', 'high')]), ('bottom_left', [('haze', 'medium')]), ('bottom_right', [('haze', 'medium')])]
top_left [('haze', 'medium')]
top_right [('haze', 'high')]
bottom_left [('haze', 'medium')]
bottom_right [('haze', 'medium')]
Processing image 2/3
[('top_left', [('jpeg compression artifact', 'high')]), ('top_right', [('jpeg compression artifact', 'high')]), ('bottom_left', [('jpeg compression artifact', 'high')]), ('bottom_right', [('jpeg compression artifact', 'high')])]
top_left [('jpeg compression artifact', 'high')]
top_right [('jpeg compression artifact', 'high')]
bottom_left [('jpeg compression artifact', 'high')]
bottom_right [('jpeg compression artifact', 'high')]
Processing image 3/3
[('top_left', [('rain', 'very high')]), ('top_right', [('rain', 'very high')]), ('bottom_left', [('rain', 'very high')]), ('bottom_right', [('rain', 'very high')])]
top_left [('rain', 'very high')]
top_right [('rain', 'very hig

In [5]:
with open("/data/zkl/AgenticIR/exploration_self_evolve/instructions_list/extra_tools_instruction.json", "w") as f:
    json.dump(instruction_list, f, indent=4)